In [23]:
try:
    import cirq
except ImportError:
    print("installing cirq...")
    !pip install cirq --quiet
    import cirq
    print("installed cirq.")

print("Libraries Successfully Imported")

Libraries Successfully Imported


In [26]:
class AncillaPropagation:
    """Tracks Pauli errors as they move through the circuit."""

    def __init__(self, data_qubits, ancilla):
        # Save the data qubits and ancilla used in the circuit
        self.data_qubits = list(data_qubits)
        self.ancilla = ancilla
        self.qubits = self.data_qubits + [self.ancilla]

        # Start every qubit with no error
        self.errors = {qubit: "I" for qubit in self.qubits}

    def reset_errors(self):
        """Resets all qubits to no error."""

        self.errors = {qubit: "I" for qubit in self.qubits}

    def combine_errors(self, current_error, new_error):
        """Combines two Pauli errors while ignoring phase."""

        # Identity does not change the other error
        if current_error == "I":
            return new_error

        if new_error == "I":
            return current_error

        # Two matching Pauli errors cancel
        if current_error == new_error:
            return "I"

        # Different Pauli errors combine into the third Pauli
        if {current_error, new_error} == {"X", "Z"}:
            return "Y"

        if {current_error, new_error} == {"X", "Y"}:
            return "Z"

        if {current_error, new_error} == {"Y", "Z"}:
            return "X"

        raise ValueError(f"Unsupported Pauli combination: {current_error} and {new_error}")

    def inject_ancilla_error(self, error):
        """Adds an X, Y, or Z error to the ancilla."""

        if error not in {"X", "Y", "Z"}:
            raise ValueError("Ancilla error must be X, Y, or Z.")

        current_error = self.errors[self.ancilla]
        self.errors[self.ancilla] = self.combine_errors(current_error, error)

    def show_errors(self):
        """Prints the current error on each qubit."""

        print("current errors:")

        for qubit in self.qubits:
            print(f"{qubit}: {self.errors[qubit]}")

    def propagate_hadamard(self, qubit):
        """Updates an error after it passes through a Hadamard gate."""

        current_error = self.errors[qubit]

        # A Hadamard switches X and Z, while Y stays Y when phase is ignored
        if current_error == "X":
            self.errors[qubit] = "Z"
        elif current_error == "Z":
            self.errors[qubit] = "X"
        elif current_error == "Y":
            self.errors[qubit] = "Y"

    def propagate_cnot(self, control, target):
        """Updates errors after they pass through a CNOT gate."""

        # Save both errors before changing either qubit
        control_error = self.errors[control]
        target_error = self.errors[target]

        # An X part on the control spreads to the target
        if control_error in {"X", "Y"}:
            self.errors[target] = self.combine_errors(target_error, "X")

        # A Z part on the target spreads back to the control
        if target_error in {"Z", "Y"}:
            self.errors[control] = self.combine_errors(control_error, "Z")

    def propagate_gate(self, operation):
        """Uses the correct propagation rule for each circuit operation."""

        gate = operation.gate
        qubits = operation.qubits

        if isinstance(gate, cirq.HPowGate):
            self.propagate_hadamard(qubits[0])

        elif isinstance(gate, cirq.CXPowGate):
            control = qubits[0]
            target = qubits[1]
            self.propagate_cnot(control, target)

        elif isinstance(gate, cirq.MeasurementGate):
            # Measurement does not spread the error to another qubit
            pass

        elif isinstance(gate, (cirq.XPowGate, cirq.ZPowGate)):
            # X and Z gates do not change the tracked Pauli label when phase is ignored
            pass

        else:
            raise NotImplementedError(f"Propagation is not implemented for {gate}.")

    def run_with_ancilla_error(self, circuit, fault_after_step, error):
        """Runs through the circuit and adds an ancilla fault after one operation."""

        self.reset_errors()

        print("initial errors:")
        self.show_errors()

        step = 1

        for moment in circuit:
            for operation in moment.operations:
                print()
                print(f"step: {step}")
                print(f"gate: {operation}")

                # Move existing errors through the current gate
                self.propagate_gate(operation)

                # Add the ancilla error after the selected step
                if step == fault_after_step:
                    print()
                    print(f"injecting {error} error on ancilla {self.ancilla}")
                    self.inject_ancilla_error(error)

                self.show_errors()
                step += 1

        return self.get_data_errors()

    def get_data_errors(self):
        """Returns only the data qubits that have an error."""

        return {qubit: self.errors[qubit] for qubit in self.data_qubits if self.errors[qubit] != "I"}

    def count_data_errors(self):
        """Counts how many data qubits have an error."""

        return len(self.get_data_errors())

In [32]:
# Create three data qubits and one ancilla
data_qubits = cirq.LineQubit.range(3)
ancilla = cirq.LineQubit(3)

# Build a circuit where the ancilla interacts with all three data qubits
baseline_circuit = cirq.Circuit(
    cirq.H(ancilla),
    cirq.CNOT(ancilla, data_qubits[0]),
    cirq.CNOT(ancilla, data_qubits[1]),
    cirq.CNOT(ancilla, data_qubits[2]),
    cirq.H(ancilla),
    cirq.measure(ancilla, key="syndrome")
)

print("baseline circuit:")
print(baseline_circuit)

baseline circuit:
0: ───────X───────────────────────────────
          │
1: ───────┼───X───────────────────────────
          │   │
2: ───────┼───┼───X───────────────────────
          │   │   │
3: ───H───@───@───@───H───M('syndrome')───


In [33]:
# Create the error propagation tracker
propagation = AncillaPropagation(data_qubits=data_qubits, ancilla=ancilla)

# Add an X error after the first Hadamard so it spreads through all three CNOTs
data_errors = propagation.run_with_ancilla_error(
    circuit=baseline_circuit,
    fault_after_step=1,
    error="X"
)

print()
print("final data errors:")
print(data_errors)

initial errors:
current errors:
q(0): I
q(1): I
q(2): I
q(3): I

step: 1
gate: H(q(3))

injecting X error on ancilla q(3)
current errors:
q(0): I
q(1): I
q(2): I
q(3): X

step: 2
gate: CNOT(q(3), q(0))
current errors:
q(0): X
q(1): I
q(2): I
q(3): X

step: 3
gate: CNOT(q(3), q(1))
current errors:
q(0): X
q(1): X
q(2): I
q(3): X

step: 4
gate: CNOT(q(3), q(2))
current errors:
q(0): X
q(1): X
q(2): X
q(3): X

step: 5
gate: H(q(3))
current errors:
q(0): X
q(1): X
q(2): X
q(3): Z

step: 6
gate: cirq.MeasurementGate(1, cirq.MeasurementKey(name='syndrome'), ())(q(3))
current errors:
q(0): X
q(1): X
q(2): X
q(3): Z

final data errors:
{cirq.LineQubit(0): 'X', cirq.LineQubit(1): 'X', cirq.LineQubit(2): 'X'}


In [40]:
data_qubits = cirq.LineQubit.range(7)
ancilla = cirq.LineQubit(7)

baseline_circuit = cirq.Circuit(
    cirq.H(ancilla),
    cirq.CNOT(ancilla, data_qubits[3]),
    cirq.CNOT(ancilla, data_qubits[4]),
    cirq.CNOT(ancilla, data_qubits[5]),
    cirq.CNOT(ancilla, data_qubits[6]),
    cirq.H(ancilla),
    cirq.measure(ancilla, key="syndrome")
)

print("baseline circuit:")
print(baseline_circuit)

baseline circuit:
3: ───────X───────────────────────────────────
          │
4: ───────┼───X───────────────────────────────
          │   │
5: ───────┼───┼───X───────────────────────────
          │   │   │
6: ───────┼───┼───┼───X───────────────────────
          │   │   │   │
7: ───H───@───@───@───@───H───M('syndrome')───


In [41]:
# Create the error propagation tracker
propagation = AncillaPropagation(data_qubits=data_qubits, ancilla=ancilla)

# Add an X error after the first Hadamard so it spreads through all three CNOTs
data_errors = propagation.run_with_ancilla_error(
    circuit=baseline_circuit,
    fault_after_step=1,
    error="X"
)

print()
print("final data errors:")
print(data_errors)

initial errors:
current errors:
q(0): I
q(1): I
q(2): I
q(3): I
q(4): I
q(5): I
q(6): I
q(7): I

step: 1
gate: H(q(7))

injecting X error on ancilla q(7)
current errors:
q(0): I
q(1): I
q(2): I
q(3): I
q(4): I
q(5): I
q(6): I
q(7): X

step: 2
gate: CNOT(q(7), q(3))
current errors:
q(0): I
q(1): I
q(2): I
q(3): X
q(4): I
q(5): I
q(6): I
q(7): X

step: 3
gate: CNOT(q(7), q(4))
current errors:
q(0): I
q(1): I
q(2): I
q(3): X
q(4): X
q(5): I
q(6): I
q(7): X

step: 4
gate: CNOT(q(7), q(5))
current errors:
q(0): I
q(1): I
q(2): I
q(3): X
q(4): X
q(5): X
q(6): I
q(7): X

step: 5
gate: CNOT(q(7), q(6))
current errors:
q(0): I
q(1): I
q(2): I
q(3): X
q(4): X
q(5): X
q(6): X
q(7): X

step: 6
gate: H(q(7))
current errors:
q(0): I
q(1): I
q(2): I
q(3): X
q(4): X
q(5): X
q(6): X
q(7): Z

step: 7
gate: cirq.MeasurementGate(1, cirq.MeasurementKey(name='syndrome'), ())(q(7))
current errors:
q(0): I
q(1): I
q(2): I
q(3): X
q(4): X
q(5): X
q(6): X
q(7): Z

final data errors:
{cirq.LineQubit(3): 'X', cir